# NEXUS Training Notebook — Google Colab

**Run all cells top-to-bottom.**

| Cell | What it does |
|---|---|
| 1 | Clone / pull the repo |
| 2 | Install dependencies + download ATTNSOM dataset |
| 3 | Environment diagnostic (GPU / RAM) |
| 4 | Single-molecule smoke test (fast forward pass) |
| 5 | 16-molecule training run — inline Python, Colab-safe settings |

All memory-sensitive parameters (quantum grid resolution, chunk size, torch.compile, dynamics mode)
are set directly in Cell 5 — no dependency on `train.py` flags.

In [ ]:
import os, subprocess

REPO = 'https://github.com/Nikku03/enzyme_Software.git'
REPO_DIR = '/content/enzyme_Software'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'main'], check=True)

os.chdir(REPO_DIR)
print('repo:', os.getcwd())

In [ ]:
!bash scripts/setup_colab_nexus.sh /content/enzyme_Software

In [ ]:
import torch, psutil

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU  : {props.name}")
    print(f"VRAM : {props.total_memory / 1024**3:.1f} GB")
else:
    print("GPU  : not available — running on CPU (will be slower)")

ram = psutil.virtual_memory()
print(f"RAM  : {ram.total / 1024**3:.1f} GB total,  {ram.available / 1024**3:.1f} GB free")
print(f"Torch: {torch.__version__}")

In [ ]:
!export PYTORCH_ALLOC_CONF=expandable_segments:True && \
python scripts/colab_smoke_test.py \
  --sdf data/ATTNSOM/cyp_dataset/3A4.sdf \
  --sample-index -1 \
  --steps 1 \
  --integration-resolution 8 \
  --integration-chunk-size 128 \
  --scan-n-points 8 \
  --scan-radius 1.0 \
  --scan-query-chunk-size 4

In [ ]:
import os, sys, json
from pathlib import Path
import torch
from torch.utils.data import DataLoader

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

REPO_DIR = "/content/enzyme_Software"
SDF      = Path(REPO_DIR) / "data/ATTNSOM/cyp_dataset/3A4.sdf"

# ── tunables ───────────────────────────────────────────────────────────────
MAX_SAMPLES       = 16    # molecules to train on
EPOCHS            = 1
STEPS             = 1     # dynamics rollout steps
INTEGRATION_RES   = 8     # 8^3 = 512 grid pts (default 16 = 4096, too slow for Colab)
INTEGRATION_CHUNK = 128   # chunk size for quantum integrator (default 1024)
# ──────────────────────────────────────────────────────────────────────────

from nexus.data.metabolic_dataset import ZaretzkiMetabolicDataset, geometric_collate_fn
from nexus.training.causal_trainer import Metabolic_Causal_Trainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}\n")

if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)

# ── dataset ────────────────────────────────────────────────────────────────
dataset = ZaretzkiMetabolicDataset(SDF, max_molecules=MAX_SAMPLES)
loader  = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=geometric_collate_fn,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)
print(f"Dataset : {len(dataset)} molecules")

# ── trainer ────────────────────────────────────────────────────────────────
trainer = Metabolic_Causal_Trainer(
    dynamics_steps=STEPS,
    dynamics_dt=0.001,
    checkpoint_dynamics=False,    # saves activation memory
    enable_wsd_scheduler=True,
    low_memory_train_mode=True,   # skip full CliffordLie rollout
    enable_static_compile=False,  # torch.compile is unsafe on Colab
).to(device)

# Colab-safe quantum grid (set AFTER .to(device))
qe = trainer.model.module1.field_engine.quantum_enforcer
qe.integration_resolution = INTEGRATION_RES
qe.integration_chunk_size = INTEGRATION_CHUNK
print(f"Quantum grid : {INTEGRATION_RES}^3 = {INTEGRATION_RES**3} pts,  chunk={INTEGRATION_CHUNK}")

trainer.set_total_training_steps(EPOCHS * max(len(loader), 1))
trainer.configure_optimizers()
print("Optimizer ready.\n")

# ── device transfer helper ─────────────────────────────────────────────────
def _to(obj, dev):
    if torch.is_tensor(obj):   return obj.to(dev)
    if isinstance(obj, dict):  return {k: _to(v, dev) for k, v in obj.items()}
    if isinstance(obj, list):  return [_to(v, dev) for v in obj]
    if isinstance(obj, tuple): return tuple(_to(v, dev) for v in obj)
    return obj

# ── training loop ──────────────────────────────────────────────────────────
history = []
for epoch in range(EPOCHS):
    reducer = {}
    trainer.train(True)
    total = len(loader)
    for i, batch in enumerate(loader, 1):
        batch = _to(batch, device)
        m = trainer.training_step(batch)
        for k, v in m.items():
            val = float(v.detach().cpu().item()) if torch.is_tensor(v) else float(v)
            reducer.setdefault(k, []).append(val)
        running = {k: sum(vs) / len(vs) for k, vs in reducer.items()}
        print(
            f"epoch={epoch+1}  batch={i}/{total}"
            f"  loss={running.get('loss_total', float('nan')):.4g}"
            f"  pred_rate={running.get('pred_rate', float('nan')):.4g}"
            f"  dag_loss={running.get('dag_causal_loss', float('nan')):.4g}",
            flush=True,
        )
    metrics = {k: sum(vs) / len(vs) for k, vs in reducer.items()}
    history.append(metrics)
    print(f"\n── epoch {epoch+1} done: {metrics}\n")

if device.type == "cuda":
    peak_mb = torch.cuda.max_memory_allocated(device) / 1024**2
    print(f"Peak GPU memory : {peak_mb:.1f} MB")

out = Path(REPO_DIR) / "colab_train_metrics.json"
out.write_text(json.dumps(history, indent=2))
print(f"Metrics saved -> {out}")